In [4]:
import pandas as pd
import numpy as np

# Load the data
file_path = 'C:/Users/prajv/Projects/MAHE/nse_combined_stocks_yfinance_data.csv'
df = pd.read_csv(file_path)

# Extract unique tickers and price types from column names
tickers = set()
price_types = set()

for col in df.columns:
    split_col = col.split('_')
    if len(split_col) == 2:  # Only consider columns with the format 'ticker_type'
        tickers.add(split_col[0])
        price_types.add(split_col[1])

# Initialize an empty dictionary to store the features
features_dict = {}

# Calculate features for each ticker
for ticker in tickers:
    # Extract relevant columns for the ticker
    close_col = f"{ticker}_Close"
    high_col = f"{ticker}_High"
    low_col = f"{ticker}_Low"
    volume_col = f"{ticker}_Volume"
    
    # Ensure the ticker has all required columns
    if all(col in df.columns for col in [close_col, high_col, low_col, volume_col]):
        # Create a DataFrame for the ticker
        ticker_df = df[[close_col, high_col, low_col, volume_col]].copy()
        
        # Calculate moving averages
        ticker_df[f'{ticker}_SMA_50'] = ticker_df[close_col].rolling(window=50).mean()
        ticker_df[f'{ticker}_SMA_200'] = ticker_df[close_col].rolling(window=200).mean()
        
        # Calculate momentum indicators (e.g., Rate of Change)
        ticker_df[f'{ticker}_ROC_10'] = ticker_df[close_col].pct_change(periods=10)
        
        # Calculate volatility (e.g., Rolling Standard Deviation)
        ticker_df[f'{ticker}_Volatility_10'] = ticker_df[close_col].rolling(window=10).std()
        
        # Store the features in the dictionary
        features_dict[ticker] = ticker_df.drop(columns=[close_col, high_col, low_col, volume_col])

# Concatenate all the features into the original DataFrame
for ticker, feature_df in features_dict.items():
    df = pd.concat([df, feature_df], axis=1)

# Handle missing values
df.fillna(method='ffill', inplace=True)
df.fillna(method='bfill', inplace=True)

# Save the DataFrame with the features
df.to_csv('C:/Users/prajv/Projects/MAHE/nse_stocks_with_features.csv', index=False)


In [5]:
df.head()

,Date,TCS.NS_Open,TCS.NS_High,TCS.NS_Low,TCS.NS_Close,TCS.NS_Volume,TCS.NS_Dividends,TCS.NS_Stock Splits,INFY.NS_Open,INFY.NS_High,...,TATAPOWER.NS_ROC_10,TATAPOWER.NS_Volatility_10,MAHLIFE.NS_SMA_50,MAHLIFE.NS_SMA_200,MAHLIFE.NS_ROC_10,MAHLIFE.NS_Volatility_10,LUPIN.NS_SMA_50,LUPIN.NS_SMA_200,LUPIN.NS_ROC_10,LUPIN.NS_Volatility_10
0,2012-04-02 00:00:00+05:30,457.467281,468.806214,457.467281,466.948975,2326988,0.0,0.0,269.708759,272.039517,...,0.065967,1.214828,85.663642,99.111909,0.006636,0.940713,504.213897,531.541256,0.027858,10.783521
1,2012-04-03 00:00:00+05:30,469.275405,472.071048,459.539601,460.595276,2269154,0.0,0.0,269.925356,270.994195,...,0.065967,1.214828,85.663642,99.111909,0.006636,0.940713,504.213897,531.541256,0.027858,10.783521
2,2012-04-04 00:00:00+05:30,457.076270,465.209041,456.294275,460.771179,2165054,0.0,0.0,268.296187,270.739947,...,0.065967,1.214828,85.663642,99.111909,0.006636,0.940713,504.213897,531.541256,0.027858,10.783521
3,2012-04-09 00:00:00+05:30,457.467082,464.505037,452.872861,455.160187,1576628,0.0,0.0,267.213195,269.270864,...,0.065967,1.214828,85.663642,99.111909,0.006636,0.940713,504.213897,531.541256,0.027858,10.783521
4,2012-04-10 00:00:00+05:30,456.020525,459.676343,450.526990,452.071411,1532992,0.0,0.0,267.636959,268.169022,...,0.065967,1.214828,85.663642,99.111909,0.006636,0.940713,504.213897,531.541256,0.027858,10.783521


In [6]:
# Load data
df = pd.read_csv('C:/Users/prajv/Projects/MAHE/nse_stocks_with_features.csv')

# List to hold the names of the target return columns
target_columns = []

# Iterate over columns to calculate target returns
for column in df.columns:
    if '_Close' in column:
        # Calculate percentage change for the current ticker's Close price
        ticker_name = column.split('_')[0]
        target_column_name = f'{ticker_name}_target_returns'
        
        # Create the target returns column and shift to align with future returns
        df[target_column_name] = df[column].pct_change().shift(-1)
        
        # Append the new column name to the list
        target_columns.append(target_column_name)

# Drop NaN values introduced by the pct_change and shift operations
df.dropna(inplace=True)

# Print the resulting DataFrame's columns to verify
print(df.columns)

# Save the DataFrame with the new target returns columns
df.to_csv('C:/Users/prajv/Projects/MAHE/stocks_with_features_and_target_returns.csv', index=False)

Index(['Date', 'TCS.NS_Open', 'TCS.NS_High', 'TCS.NS_Low', 'TCS.NS_Close',
       'TCS.NS_Volume', 'TCS.NS_Dividends', 'TCS.NS_Stock Splits',
       'INFY.NS_Open', 'INFY.NS_High',
       ...
       'DLF.NS_target_returns', 'GODREJPROP.NS_target_returns',
       'OBEROIRLTY.NS_target_returns', 'PHOENIXLTD.NS_target_returns',
       'PRESTIGE.NS_target_returns', 'BRIGADE.NS_target_returns',
       'SOBHA.NS_target_returns', 'IBREALEST.NS_target_returns',
       'PURVA.NS_target_returns', 'MAHLIFE.NS_target_returns'],
      dtype='object', length=902)
